In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

2026-04-25 16:53:30.101623: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-25 16:53:31.523469: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


TensorFlow: 2.12.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-04-25 16:53:35.092972: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-25 16:53:35.164058: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-25 16:53:35.164179: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [2]:
DATASET_PATH = "/mnt/d/waste-classification-system-II/dataset/New Trash Classfication Dataset/new-dataset-trash-type-v2"

image_paths = []
labels = []

class_names = sorted([
    cls for cls in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, cls))
])

for label, cls in enumerate(class_names):
    class_path = os.path.join(DATASET_PATH, cls)

    for img_name in os.listdir(class_path):
        if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(class_path, img_name))
            labels.append(label)

print("Classes:", class_names)
print("Total classes:", len(class_names))
print("Total images:", len(image_paths))

print("\nClass distribution:")
for label, cls in enumerate(class_names):
    print(cls, ":", labels.count(label))

Classes: ['cardboard', 'e-waste', 'glass', 'metal', 'organic', 'paper', 'plastic', 'textile', 'trash']
Total classes: 9
Total images: 8407

Class distribution:
cardboard : 893
e-waste : 993
glass : 948
metal : 901
organic : 967
paper : 853
plastic : 891
textile : 985
trash : 976


In [3]:
idx = 0
img = tf.io.read_file(image_paths[idx])
img = tf.image.decode_image(img, channels=3)

print("Sample path:", image_paths[idx])
print("Image shape:", img.shape)
print("Label:", labels[idx], "->", class_names[labels[idx]])


Sample path: /mnt/d/waste-classification-system-II/dataset/New Trash Classfication Dataset/new-dataset-trash-type-v2/cardboard/cardboard1.jpg
Image shape: (384, 512, 3)
Label: 0 -> cardboard


2026-04-25 16:46:40.831931: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-25 16:46:40.832088: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-25 16:46:40.832170: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-25 16:46:41.036931: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-25 16:46:41.037131: I tensorflow/compile

In [5]:
train_paths, temp_paths, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.20, random_state=42, stratify=labels
)

val_paths, test_paths, y_val, y_test = train_test_split(
    temp_paths, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

Train: 6725
Validation: 841
Test: 841


In [ ]:
print("Train:", Counter(y_train))
print("Validation:", Counter(y_val))
print("Test:", Counter(y_test))

Train: Counter({1: 794, 7: 788, 8: 781, 4: 774, 2: 758, 3: 721, 0: 714, 6: 713, 5: 682})
Validation: Counter({1: 100, 7: 98, 8: 98, 4: 96, 2: 95, 0: 90, 3: 90, 6: 89, 5: 85})
Test: Counter({1: 99, 7: 99, 4: 97, 8: 97, 2: 95, 3: 90, 0: 89, 6: 89, 5: 86})


In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 4

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label



In [9]:
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, y_train))
train_ds = train_ds.map(load_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, y_val))
val_ds = val_ds.map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, y_test))
test_ds = test_ds.map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Dataset pipeline ready")

Dataset pipeline ready


In [10]:
for images, labels_batch in train_ds.take(1):
    print(images.shape)
    print(labels_batch.shape)

2026-04-25 16:47:52.064238: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype int32 and shape [6725]
	 [[{{node Placeholder/_1}}]]
2026-04-25 16:47:52.064683: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype int32 and shape [6725]
	 [[{{node Placeholder/_1}}]]


(8, 96, 96, 3)
(8,)


## MobileNetV2 transfer learning

In [11]:
print("GPUs:", tf.config.list_physical_devices('GPU'))

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [12]:
num_classes = len(class_names)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 94, 94, 16)        448       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 47, 47, 16)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 45, 45, 32)        4640      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 22, 22, 32)       0         
 2D)                                                             
                                                                 
 global_average_pooling2d (G  (None, 32)               0         
 lobalAveragePooling2D)                                          
                                                        

In [13]:

EPOCHS = 2

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/2


2026-04-25 16:48:00.243658: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype int32 and shape [6725]
	 [[{{node Placeholder/_1}}]]
2026-04-25 16:48:00.243978: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [6725]
	 [[{{node Placeholder/_0}}]]


: 